# BigQuery + DVC Data Extraction and EDA

This notebook demonstrates how to pull the three pipeline tables from BigQuery, version them with DVC, and perform exploratory data analysis on `category_tree`, `events`, and `item_properties`.

## Setup

The notebook uses the `BigQueryQuery` helper from `ecommerce_recommender/data_pipeline/bigquery_query.py` to run queries, write CSV exports, and add them to DVC. Make sure your `.venv` is active and DVC is installed.

In [1]:
import sys
from pathlib import Path


def get_repo_root(start_path: Path) -> Path:
    for path in [start_path] + list(start_path.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not locate repository root. Run this notebook inside the repository.")

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = get_repo_root(CURRENT_DIR)
print(f"Repository root found at: {REPO_ROOT}")

sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ecommerce_recommender.data_pipeline.bigquery_query import BigQueryQuery


Repository root found at: /home/gusdpaula/code-dev/MLENG_FIAP/fase_2


## Configure the Extractor

Set the BigQuery project and dataset values, then create an output folder for the exported CSVs.

In [2]:
PROJECT_ID = "retail-rocket-498618"  # update to your project id
DATASET_ID = "retailrocket"  # update to your dataset id

OUTPUT_DIR = REPO_ROOT / "ecommerce_recommender" / "data" / "raw"
DVC_REPO_PATH = REPO_ROOT

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

query_client = BigQueryQuery(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    output_dir=OUTPUT_DIR,
    dvc_repo_path=DVC_REPO_PATH,
)


/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## Extract the Three Pipeline Tables

Export the tables to local CSV files and version them with DVC. If a CSV already exists, the helper will reuse it by default and skip re-downloading unless `force=True` is passed.

In [3]:
category_tree_csv = query_client.extract_table(
    table_name='retail-rocket_category_tree',
    destination_name='category_tree.csv',
)
events_csv = query_client.extract_table(
    table_name='retail-rocket_events',
    destination_name='events.csv',
)
item_properties_csv = query_client.extract_table(
    table_name='retail-rocket_item_properties',
    destination_name='item_properties.csv',
)

category_tree_csv, events_csv, item_properties_csv

(PosixPath('/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/category_tree.csv'),
 PosixPath('/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/events.csv'),
 PosixPath('/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/item_properties.csv'))

## Load Extracted CSVs

Read the exported files into pandas DataFrames for analysis.

In [4]:
df_category_tree = pd.read_csv(category_tree_csv)
df_events = pd.read_csv(events_csv)
df_item_properties = pd.read_csv(item_properties_csv)

print('Loaded category_tree shape:', df_category_tree.shape)
print('Loaded events shape:', df_events.shape)
print('Loaded item_properties shape:', df_item_properties.shape)

df_category_tree.head(), df_events.head(), df_item_properties.head()


Loaded category_tree shape: (1669, 2)
Loaded events shape: (2756101, 5)
Loaded item_properties shape: (19774876, 4)


/tmp/ipykernel_6817/2657188350.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_item_properties = pd.read_csv(item_properties_csv)


(   categoryid  parentid
 0         231       NaN
 1         791       NaN
 2        1490       NaN
 3         431       NaN
 4         755       NaN,
        timestamp  visitorid event  itemid  transactionid
 0  1433221955914     483717  view  253185            NaN
 1  1433223291897     794181  view  439202            NaN
 2  1433224554732     453474  view  250696            NaN
 3  1433221377547    1153198  view  388242            NaN
 4  1433224266445     273888  view  205392            NaN,
        timestamp  itemid property  \
 0  1442113200000  148046      447   
 1  1442113200000  155662       71   
 2  1442113200000  341269      915   
 3  1442113200000  298900      496   
 4  1442113200000  377947      765   
 
                                                value  
 0  1317249 440924 76225 1252024 440924 863390 125...  
 1                                             669505  
 2                        325894 1325686 72261 505888  
 3  357961 986363 357961 253573 999874 632043 

## Category Tree EDA

Inspect the hierarchical structure and count categories.

In [5]:
print('Category tree rows:', len(df_category_tree))
print('Unique category IDs:', df_category_tree['categoryid'].nunique())
print('Top parent categories:')
print(df_category_tree['parentid'].value_counts().head(10))

Category tree rows: 1669
Unique category IDs: 1669
Top parent categories:
parentid
250.0     31
1009.0    22
362.0     22
351.0     19
1259.0    18
1687.0    17
945.0     15
312.0     15
92.0      13
1482.0    13
Name: count, dtype: int64


## Events EDA

Analyze event counts, unique users, and event types.

In [6]:
print('Total events:', len(df_events))
print('Unique users:', df_events['visitorid'].nunique())

if 'event' in df_events.columns:
    print('Event type counts:')
    print(df_events['event'].value_counts().head(10))
else:
    print('No event type column in events table.')

Total events: 2756101
Unique users: 1407580
Event type counts:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


## Item Properties EDA

Understand how many unique items and properties exist in the merged properties table.

In [7]:
print('Total item properties rows:', len(df_item_properties))
print('Unique items:', df_item_properties['itemid'].nunique())
print('Sample properties:')
print(df_item_properties.head(10))

Total item properties rows: 19774876
Unique items: 417053
Sample properties:
       timestamp  itemid property  \
0  1442113200000  148046      447   
1  1442113200000  155662       71   
2  1442113200000  341269      915   
3  1442113200000  298900      496   
4  1442113200000  377947      765   
5  1442113200000  287345      620   
6  1440298800000  159652      851   
7  1440298800000  313457      837   
8  1440298800000  272597      207   
9  1440298800000  198623     1020   

                                               value  
0  1317249 440924 76225 1252024 440924 863390 125...  
1                                             669505  
2                        325894 1325686 72261 505888  
3  357961 986363 357961 253573 999874 632043 8664...  
4  488682 1057641 885495 1186729 115283 1186729 4...  
5                664227 1305534 664227 463202 664227  
6                                             769062  
7                                             769062  
8                   

## Notes

- The exported CSV files are versioned with DVC through `dvc add`.
- If the dataset is large, use filtered queries or `LIMIT` to reduce output size for exploratory analysis.
- Update `PROJECT_ID` and `DATASET_ID` to match your environment before running.